In [1]:
import torch 
from torch import nn


In [2]:
x = torch.randn(8, 16, 28, 28)
def simple_batch_norm(x,eps=1e-5):
    gemma = 1
    beta = 0
    chl_mean = x.mean(dim=(0, 2, 3), keepdim=True)
    chl_var = x.var(dim=(0,2,3),correction = 0,keepdim = True)
    return gemma*(x-chl_mean)/(torch.sqrt(chl_var+eps))+beta

In [13]:
class MyBatchNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn. Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        self.eps = eps
        self.momentum = momentum

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=(0, 2, 3))
            var = x.var(dim=(0, 2, 3), correction=0)

            with torch.no_grad():
                unbiased_var = x.var(
                    dim=(0, 2, 3),
                    correction=1
                )

                self.running_mean.mul_(
                    1 - self.momentum
                ).add_(
                    mean, alpha=self.momentum
                )

                self.running_var.mul_(
                    1 - self.momentum
                ).add_(
                    unbiased_var, alpha=self.momentum
                )
        else:
            mean = self.running_mean
            var = self.running_var

        mean = mean.view(1, -1, 1, 1)
        var = var.view(1, -1, 1, 1)

        gamma = self.gamma.view(1, -1, 1, 1)
        beta = self.beta.view(1, -1, 1, 1)

        x_hat = (x - mean) / torch.sqrt(var + self.eps)

        return gamma * x_hat + beta

In [14]:
torch.manual_seed(42)

custom = MyBatchNorm2d(3)
official = nn.BatchNorm2d(3)

x = torch.randn(4, 3, 5, 5)

# 测试训练模式
custom.train()
official.train()

y_custom = custom(x)
y_official = official(x)

print("Train output:",
      torch.allclose(y_custom, y_official, atol=1e-5))

print("Running mean:",
      torch.allclose(custom.running_mean,
                     official.running_mean, atol=1e-5))

print("Running var:",
      torch.allclose(custom.running_var,
                     official.running_var, atol=1e-5))

# 测试评估模式：换一批不同分布的数据
custom.eval()
official.eval()

x_test = torch.randn(2, 3, 5, 5) * 4 + 5

with torch.no_grad():
    y_custom = custom(x_test)
    y_official = official(x_test)

print("Eval output:",
      torch.allclose(y_custom, y_official, atol=1e-5))

print("Parameters:",
      sum(p.numel() for p in custom.parameters()))

Train output: True
Running mean: True
Running var: True
Eval output: True
Parameters: 6


In [15]:
torch.manual_seed(42)

custom = MyBatchNorm2d(3)
official = nn.BatchNorm2d(3)

x1 = torch.randn(4, 3, 5, 5, requires_grad=True)
x2 = x1.detach().clone().requires_grad_()

y1 = custom(x1)
y2 = official(x2)

g = torch.randn_like(y1)

(y1 * g).sum().backward()
(y2 * g).sum().backward()

print("Input grad:",
      torch.allclose(x1.grad, x2.grad, atol=1e-5))

print("Gamma grad:",
      torch.allclose(custom.gamma.grad,
                     official.weight.grad, atol=1e-5))

print("Beta grad:",
      torch.allclose(custom.beta.grad,
                     official.bias.grad, atol=1e-5))

Input grad: True
Gamma grad: True
Beta grad: True
